In [1]:
import sys
sys.path.insert(0, "..")
from src.spark_utils import get_spark
from pyspark.sql import functions as F

spark = get_spark("03-features")
df = spark.read.parquet("../data/yellow_2016_03_clean.parquet")
print(f"Loaded: {df.count():,} rows")

Loaded: 7,937,904 rows


In [ ]:
df = (df
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_dayofweek", F.dayofweek("tpep_pickup_datetime"))  # 1=Sun..7=Sat
    .withColumn("pickup_day", F.dayofmonth("tpep_pickup_datetime"))
    .withColumn("is_weekend", F.col("pickup_dayofweek").isin([1, 7]).cast("int"))
    .withColumn("is_rush_hour", ((F.col("pickup_hour").between(7, 9)) | (F.col("pickup_hour").between(16, 19))).cast("int"))
    .withColumn("is_night", ((F.col("pickup_hour") >= 22) | (F.col("pickup_hour") <= 5)).cast("int"))
)

In [ ]:
df = df.withColumn(
    "avg_speed_mph",
    F.when(F.col("trip_duration_min") > 0,
           F.col("trip_distance") / (F.col("trip_duration_min") / 60.0))
     .otherwise(0.0)
)
# NYC taxis don't do 100mph
df = df.filter(F.col("avg_speed_mph") < 100)

In [4]:
feature_cols = [
    "passenger_count", "trip_distance", "fare_amount",
    "extra", "mta_tax", "tolls_amount",
    "trip_duration_min", "avg_speed_mph",
    "pickup_hour", "pickup_dayofweek", "pickup_day",
    "is_weekend", "is_rush_hour", "is_night",
    "VendorID", "RatecodeID",
]
target_col = "tip_amount"

df_features = df.select(*feature_cols, target_col)
df_features.printSchema()
df_features.show(5)

root
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- trip_duration_min: double (nullable = true)
 |-- avg_speed_mph: double (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- pickup_dayofweek: integer (nullable = true)
 |-- pickup_day: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- is_rush_hour: integer (nullable = true)
 |-- is_night: integer (nullable = true)
 |-- VendorID: integer (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- tip_amount: double (nullable = true)

+---------------+-------------+-----------+-----+-------+------------+------------------+------------------+-----------+----------------+----------+----------+------------+--------+--------+----------+----------+
|passenger_count|trip_distance|fare_amou

In [5]:
import pandas as pd

correlations = []
for col in feature_cols:
    c = df_features.stat.corr(col, target_col)
    correlations.append((col, round(c, 4)))

corr_df = pd.DataFrame(correlations, columns=["feature", "corr_with_tip"])
corr_df = corr_df.reindex(corr_df.corr_with_tip.abs().sort_values(ascending=False).index)
print(corr_df.to_string(index=False))
corr_df.to_csv("../results/feature_correlations.csv", index=False)

          feature  corr_with_tip
      fare_amount         0.8189
    trip_distance         0.7985
trip_duration_min         0.7161
     tolls_amount         0.5949
    avg_speed_mph         0.4680
       RatecodeID         0.3903
            extra         0.0985
          mta_tax        -0.0676
       is_weekend        -0.0351
     is_rush_hour        -0.0212
         is_night         0.0200
      pickup_hour         0.0186
 pickup_dayofweek        -0.0180
         VendorID         0.0149
  passenger_count         0.0091
       pickup_day         0.0054


In [6]:
(df_features.write
   .mode("overwrite")
   .parquet("../data/yellow_2016_03_features.parquet"))
print(f"Features written: {df_features.count():,} rows, {len(feature_cols)} features")
spark.stop()

Features written: 7,937,686 rows, 16 features
